# Conceptos Introductorios — Aprendizaje No Supervisado

_Glosario interactivo con definiciones, ejemplos visuales y casos de uso_

**Módulo 2 — Aprendizaje No Supervisado** | DSRP Machine Learning Engineering  
**Profesor:** Miguel Arquez

---

> 📖 **Propósito:** Este notebook sirve como referencia rápida de los conceptos clave del módulo 2. Cada término incluye:
> - Definición clara y concisa
> - Fórmula matemática (cuando aplica)
> - Ejemplo visual o código demostrativo
> - Casos de uso prácticos

![Aprendizaje No Supervisado](assets/header.png)

## Índice de Conceptos

### 1. Fundamentos
- [Aprendizaje No Supervisado](#aprendizaje-no-supervisado)
- [Estructura en los Datos](#estructura-en-los-datos)
- [Métricas Internas vs Externas](#metricas-internas-vs-externas)

### 2. Clustering
- [Clustering](#clustering)
- [K-Means](#k-means)
- [Inercia](#inercia)
- [Clustering Jerárquico](#clustering-jerarquico)
- [Dendrograma](#dendrograma)
- [Linkage](#linkage)

### 3. Métricas de Distancia
- [Distancia Euclidiana](#distancia-euclidiana)
- [Distancia de Manhattan](#distancia-de-manhattan)
- [Distancia del Coseno](#distancia-del-coseno)

### 4. Evaluación de Clustering
- [Coeficiente de Silueta (Silhouette)](#coeficiente-de-silueta)
- [Método del Codo (Elbow)](#metodo-del-codo)
- [Davies-Bouldin Index](#davies-bouldin-index)
- [Calinski-Harabasz Index](#calinski-harabasz-index)

### 5. Reducción de Dimensionalidad
- [PCA (Principal Component Analysis)](#pca)
- [Componentes Principales](#componentes-principales)
- [Varianza Explicada](#varianza-explicada)
- [Loadings](#loadings)
- [Scree Plot](#scree-plot)
- [Maldición de la Dimensión](#maldicion-de-la-dimension)

### 6. Reglas de Asociación
- [Market Basket Analysis](#market-basket-analysis)
- [Soporte (Support)](#soporte)
- [Confianza (Confidence)](#confianza)
- [Lift](#lift)
- [Algoritmo Apriori](#algoritmo-apriori)
- [FP-Growth](#fp-growth)

---

## 1. Fundamentos

### <a id="aprendizaje-no-supervisado"></a>Aprendizaje No Supervisado

**Definición:** Familia de técnicas de machine learning donde **no tenemos etiquetas** $y$. Solo disponemos de observaciones $\{x_1, x_2, \dots, x_n\}$ y queremos descubrir **estructura oculta** en los datos.

**Diferencia con supervisado:**

| Aspecto | Supervisado | No Supervisado |
|---------|-------------|----------------|
| **Datos** | $(X, y)$ — tenemos etiquetas | $X$ — sin etiquetas |
| **Objetivo** | Predecir $y$ para nuevos $x$ | Descubrir patrones, grupos, ejes |
| **Pregunta** | "¿Cuánto vale?" "¿De qué clase es?" | "¿Qué grupos hay?" "¿Qué ejes resumen los datos?" |
| **Evaluación** | Objetiva (MAE, F1, AUC) | Subjetiva (silhouette, varianza explicada, validación de negocio) |
| **Ejemplos** | Regresión, clasificación | Clustering, PCA, reglas de asociación |

**Casos de uso:**
- Segmentación de clientes (marketing)
- Detección de anomalías (fraude, fallas)
- Compresión y visualización de datos de alta dimensión
- Descubrimiento de patrones de compra (retail)
- Agrupamiento de documentos o imágenes similares

### <a id="estructura-en-los-datos"></a>Estructura en los Datos

**Definición:** Patrones, regularidades o relaciones que existen en los datos pero que no son inmediatamente obvias. El aprendizaje no supervisado busca **revelar** esta estructura.

**Tipos de estructura:**
1. **Grupos (clusters):** Observaciones que se parecen entre sí
2. **Dimensiones latentes:** Ejes que capturan la variación principal
3. **Asociaciones:** Ítems que ocurren juntos frecuentemente
4. **Anomalías:** Observaciones que no encajan en ningún patrón

**Ejemplo:** En un dataset de clientes de telecom, la estructura podría ser:
- 3 segmentos naturales: "clientes nuevos", "clientes leales", "clientes premium"
- 2 ejes principales: "antigüedad" y "gasto mensual"
- Regla: "clientes con fibra óptica tienden a contratar streaming"

### <a id="metricas-internas-vs-externas"></a>Métricas Internas vs Externas

**Definición:** Formas de evaluar la calidad de los resultados en aprendizaje no supervisado.

**Métricas Internas:** Miden la calidad **sin usar etiquetas externas**. Solo usan la estructura de los datos.

| Métrica | Qué mide | Rango | Mejor valor |
|---------|----------|-------|-------------|
| **Silhouette** | Cohesión interna vs separación entre clusters | [-1, 1] | Cercano a 1 |
| **Inercia** | Suma de distancias al centroide (K-Means) | [0, ∞) | Menor es mejor |
| **Davies-Bouldin** | Ratio de dispersión intra-cluster vs inter-cluster | [0, ∞) | Menor es mejor |
| **Calinski-Harabasz** | Ratio de varianza entre-clusters vs intra-cluster | [0, ∞) | Mayor es mejor |

**Métricas Externas:** Comparan los clusters descubiertos con **etiquetas conocidas** (cuando las tenemos para validación).

| Métrica | Qué mide |
|---------|----------|
| **Adjusted Rand Index (ARI)** | Similitud entre dos particiones (ajustado por azar) |
| **Normalized Mutual Information (NMI)** | Información mutua normalizada |
| **Validación de negocio** | ¿Los segmentos son accionables? ¿Tienen sentido? |

> 💡 **Importante:** La validación de negocio es la más importante — un clustering estadísticamente perfecto puede ser inútil si no se traduce en acciones concretas.

---

## 2. Clustering

### <a id="clustering"></a>Clustering

**Definición:** Técnica que agrupa observaciones en $K$ clusters (grupos) tales que:
- Los puntos **dentro** de un cluster se parecen entre sí (alta cohesión interna)
- Los puntos de **distintos** clusters son diferentes (alta separación)

**Formulación matemática:**

Buscamos una partición $C_1, \dots, C_K$ que minimice la disimilitud intra-cluster:

$$
\min_{C_1,\dots,C_K} \sum_{k=1}^{K} \sum_{x_i \in C_k} d(x_i, \mu_k)
$$

donde $\mu_k$ es el centro del cluster $k$ y $d(\cdot,\cdot)$ es una métrica de distancia.

**Casos de uso:**
- **Marketing:** Segmentación de clientes para campañas personalizadas
- **Retail:** Agrupamiento de productos similares
- **Salud:** Identificación de subtipos de enfermedades
- **Biología:** Agrupamiento de genes con expresión similar
- **Visión:** Segmentación de imágenes

In [ ]:
# Ejemplo visual: K-Means sobre datos sintéticos
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# Generamos 3 clusters bien separados
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

# Aplicamos K-Means
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Datos originales (sin clustering)
axes[0].scatter(X[:, 0], X[:, 1], c='gray', s=30, alpha=0.6)
axes[0].set_title('Datos sin etiquetar')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Datos con clusters descubiertos
axes[1].scatter(X[:, 0], X[:, 1], c=kmeans.labels_, cmap='viridis', s=30, alpha=0.6)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                marker='X', c='red', s=200, edgecolor='black', label='Centroides')
axes[1].set_title('Clusters descubiertos por K-Means')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Número de clusters: {kmeans.n_clusters}')
print(f'Inercia (suma de distancias al centroide): {kmeans.inertia_:.2f}')


### <a id="k-means"></a>K-Means

**Definición:** Algoritmo de clustering que encuentra $K$ centroides $\mu_1, \dots, \mu_K$ y asigna cada punto al centroide más cercano. Minimiza la **inercia** (suma de cuadrados intra-cluster).

**Algoritmo (Lloyd):**
1. **Inicializar** $K$ centroides (idealmente con k-means++ para mejor convergencia)
2. **Asignación:** Cada $x_i$ va al cluster del centroide más cercano
3. **Actualización:** Recalcular $\mu_k$ como la media de los puntos asignados a $C_k$
4. **Repetir** pasos 2-3 hasta convergencia (centroides no se mueven)

**Función objetivo:**

$$
J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - \mu_k \|^2
$$

**Ventajas:**
- ✅ Rápido y escalable
- ✅ Fácil de implementar e interpretar
- ✅ Funciona bien con clusters esféricos y de tamaño similar

**Limitaciones:**
- ❌ Necesitas fijar $K$ de antemano
- ❌ Asume clusters esféricos (usa distancia euclidiana)
- ❌ Sensible a la inicialización (usar `n_init` alto)
- ❌ Sensible a la escala y outliers
- ❌ Solo funciona con distancia euclidiana

**Cuándo usar:**
- Tienes una idea aproximada de cuántos grupos esperas
- Los clusters son aproximadamente esféricos y de tamaño similar
- Tienes muchos datos y necesitas velocidad

### <a id="inercia"></a>Inercia

**Definición:** Suma de las distancias al cuadrado de cada punto a su centroide asignado. Es la función objetivo que K-Means minimiza.

**Fórmula:**

$$
\text{Inercia} = \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - \mu_k \|^2
$$

**Interpretación:**
- **Inercia baja:** Los puntos están muy cerca de sus centroides → clusters compactos
- **Inercia alta:** Los puntos están dispersos → clusters poco definidos

**Propiedades:**
- Siempre **disminuye** al aumentar $K$ (con $K=n$ cada punto es su propio cluster, inercia = 0)
- Por eso no se usa sola para elegir $K$ — se busca el "codo" donde la mejora se aplana

**Uso práctico:**
- Comparar diferentes inicializaciones de K-Means (elegir la de menor inercia)
- Método del codo para elegir $K$
- Monitorear convergencia del algoritmo

### <a id="clustering-jerarquico"></a>Clustering Jerárquico

**Definición:** Familia de algoritmos que construyen una **jerarquía** de clusters en forma de árbol (dendrograma). A diferencia de K-Means, **no requiere fijar $K$ de antemano**.

**Dos estrategias:**

| Estrategia | Descripción | Uso |
|------------|-------------|-----|
| **Aglomerativo (bottom-up)** | Cada punto empieza siendo su propio cluster. Se van fusionando los pares más cercanos. | **Más usado** |
| **Divisivo (top-down)** | Todos los puntos empiezan en un solo cluster. Se van dividiendo. | Menos común |

**Ventajas:**
- ✅ No necesitas fijar $K$ de antemano
- ✅ Produce un dendrograma que muestra la estructura completa
- ✅ Acepta cualquier métrica de distancia
- ✅ Determinístico (no depende de inicialización aleatoria)

**Limitaciones:**
- ❌ Más lento que K-Means (O(n² log n) vs O(nKi))
- ❌ No escala bien a millones de observaciones
- ❌ No tiene `predict()` — para asignar nuevos puntos hay que reajustar
- ❌ Sensible a outliers (especialmente con single linkage)

**Cuándo usar:**
- No sabes cuántos clusters esperar
- Quieres explorar la estructura jerárquica
- Tienes pocos datos (< 10k observaciones)
- Necesitas usar una métrica de distancia específica

### <a id="dendrograma"></a>Dendrograma

**Definición:** Diagrama en forma de árbol que muestra cómo se fusionan los clusters en el clustering jerárquico. El eje vertical representa la **distancia de fusión**.

**Cómo leerlo:**
- **Hojas (abajo):** Observaciones individuales
- **Ramas:** Fusiones de clusters
- **Altura de la rama:** Distancia a la que se fusionaron
- **Corte horizontal:** Define el número de clusters

**Cómo elegir $K$:**
1. Buscar el **mayor salto vertical** sin cruce horizontal
2. Trazar una línea horizontal a esa altura
3. El número de ramas que cruza = número de clusters

**Interpretación:**
- Ramas que se fusionan **temprano** (abajo) → observaciones muy similares
- Ramas que se fusionan **tarde** (arriba) → clusters bien separados
- Si todo se fusiona muy arriba → no hay estructura clara

In [ ]:
# Ejemplo visual: Dendrograma
from scipy.cluster.hierarchy import dendrogram, linkage

# Usamos una submuestra para que el dendrograma sea legible
np.random.seed(42)
idx = np.random.choice(len(X), size=50, replace=False)
X_sub = X[idx]

# Calculamos el linkage (matriz de fusiones)
Z = linkage(X_sub, method='ward')

# Visualización
fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, ax=ax, color_threshold=8)
ax.axhline(8, color='red', linestyle='--', label='Corte sugerido (K=3)')
ax.set_title('Dendrograma — Clustering Jerárquico Aglomerativo')
ax.set_xlabel('Índice de observación')
ax.set_ylabel('Distancia de fusión')
ax.legend()
plt.show()

print('Interpretación:')
print('- Las 3 ramas principales (colores diferentes) son los 3 clusters')
print('- La línea roja indica dónde cortar para obtener K=3 clusters')
print('- Observaciones que se fusionan temprano (abajo) son muy similares')


### <a id="linkage"></a>Linkage

**Definición:** Criterio para medir la distancia entre **clusters** (no entre puntos individuales) en el clustering jerárquico.

**Tipos de linkage:**

| Linkage | Fórmula | Comportamiento | Cuándo usar |
|---------|---------|----------------|-------------|
| **Single (mínimo)** | $\min_{a \in A, b \in B} d(a, b)$ | Tiende a clusters alargados ("efecto cadena") | Detectar formas irregulares |
| **Complete (máximo)** | $\max_{a \in A, b \in B} d(a, b)$ | Clusters compactos y esféricos | Clusters bien separados |
| **Average** | $\frac{1}{|A||B|} \sum_{a \in A, b \in B} d(a, b)$ | Balance entre single y complete | Uso general |
| **Ward** | Minimiza incremento de suma de cuadrados intra-cluster | Similar a K-Means, clusters de tamaño parecido | **Default recomendado** (con datos escalados) |

**Ejemplo visual:**

Imagina dos clusters A y B:
- **Single:** Mide la distancia entre los 2 puntos más cercanos (uno de A, uno de B)
- **Complete:** Mide la distancia entre los 2 puntos más lejanos
- **Average:** Promedio de todas las distancias entre pares
- **Ward:** Cuánto aumentaría la inercia total si fusionamos A y B

**Recomendación:** Usa **Ward** cuando los datos están escalados y esperas clusters de tamaño similar. Es el más robusto en la práctica.

---

## 3. Métricas de Distancia

### <a id="distancia-euclidiana"></a>Distancia Euclidiana

**Definición:** Distancia "en línea recta" entre dos puntos en el espacio euclidiano. Es la métrica más común en clustering.

**Fórmula:**

$$
d_{\text{euc}}(x, y) = \sqrt{\sum_{j=1}^{p} (x_j - y_j)^2} = \|x - y\|_2
$$

**Propiedades:**
- Sensible a la **escala** de las variables → siempre escalar antes de usar
- Asume que todas las dimensiones son **igualmente importantes**
- Funciona bien con datos numéricos continuos

**Cuándo usar:**
- Variables numéricas en la misma escala (o escaladas)
- Clusters esperados son esféricos
- K-Means (usa euclidiana por defecto)

**Ejemplo:**
```python
from scipy.spatial.distance import euclidean
x = [1, 2, 3]
y = [4, 5, 6]
d = euclidean(x, y)  # √((4-1)² + (5-2)² + (6-3)²) = √27 ≈ 5.20
```

### <a id="distancia-de-manhattan"></a>Distancia de Manhattan

**Definición:** Suma de las diferencias absolutas en cada dimensión. También llamada **distancia L1** o **city block distance** (como caminar en una cuadrícula de ciudad).

**Fórmula:**

$$
d_{\text{man}}(x, y) = \sum_{j=1}^{p} |x_j - y_j| = \|x - y\|_1
$$

**Propiedades:**
- Más **robusta a outliers** que la euclidiana (no eleva al cuadrado)
- Útil cuando el movimiento está restringido a ejes (como en una cuadrícula)
- Menos sensible a diferencias en una sola dimensión

**Cuándo usar:**
- Datos con outliers
- Variables en escalas muy diferentes (aunque mejor escalar)
- Problemas de optimización donde el movimiento es por ejes

**Ejemplo:**
```python
from scipy.spatial.distance import cityblock
x = [1, 2, 3]
y = [4, 5, 6]
d = cityblock(x, y)  # |4-1| + |5-2| + |6-3| = 3 + 3 + 3 = 9
```

**Comparación con Euclidiana:**
- Manhattan: suma de diferencias absolutas
- Euclidiana: raíz cuadrada de la suma de diferencias al cuadrado
- Manhattan ≥ Euclidiana (desigualdad triangular)

### <a id="distancia-del-coseno"></a>Distancia del Coseno

**Definición:** Mide el **ángulo** entre dos vectores, ignorando su magnitud. Muy usada en NLP y sistemas de recomendación.

**Fórmula:**

$$
d_{\cos}(x, y) = 1 - \frac{x \cdot y}{\|x\| \, \|y\|} = 1 - \cos(\theta)
$$

donde $\theta$ es el ángulo entre los vectores.

**Similitud del coseno:**

$$
\text{sim}_{\cos}(x, y) = \frac{x \cdot y}{\|x\| \, \|y\|} \in [-1, 1]
$$

- **1:** Vectores idénticos (ángulo 0°)
- **0:** Vectores ortogonales (ángulo 90°)
- **-1:** Vectores opuestos (ángulo 180°)

**Propiedades:**
- **Ignora la magnitud** — solo importa la dirección
- Útil cuando la escala absoluta no importa
- Invariante a multiplicación por escalar

**Cuándo usar:**
- **Text mining:** Comparar documentos (bag-of-words, TF-IDF)
- **Sistemas de recomendación:** Comparar perfiles de usuarios/productos
- **Embeddings:** Comparar vectores de alta dimensión (word2vec, etc.)

**Ejemplo:**
```python
from scipy.spatial.distance import cosine
x = [1, 2, 3]
y = [2, 4, 6]  # y = 2*x (misma dirección, distinta magnitud)
d = cosine(x, y)  # ≈ 0 (ángulo pequeño, vectores casi paralelos)
```

> 💡 **Importante:** En text mining, dos documentos con las mismas palabras pero diferentes longitudes tendrán distancia coseno pequeña (son similares en contenido).

In [ ]:
# Comparación visual de métricas de distancia
from scipy.spatial.distance import cdist

# Generamos puntos aleatorios
np.random.seed(42)
puntos = np.random.uniform(0, 10, size=(40, 2))
ref = np.array([[5, 5]])  # Punto de referencia (estrella roja)

# Calculamos distancias con cada métrica
metricas = ['euclidean', 'cityblock', 'cosine']
titulos = ['Distancia Euclidiana', 'Distancia Manhattan', 'Distancia Coseno']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric, titulo in zip(axes, metricas, titulos):
    # Calculamos distancias desde el punto de referencia
    d = cdist(ref, puntos, metric=metric).ravel()
    
    # Scatter plot coloreado por distancia
    sc = ax.scatter(puntos[:, 0], puntos[:, 1], c=d, cmap='viridis', s=60, edgecolor='white')
    ax.scatter(*ref[0], marker='*', s=400, c='red', edgecolor='black', label='Referencia')
    
    ax.set_title(titulo)
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.legend()
    plt.colorbar(sc, ax=ax, label='Distancia')

plt.tight_layout()
plt.show()

print('Interpretación:')
print('- Colores más oscuros = mayor distancia al punto rojo')
print('- Euclidiana: círculos concéntricos (isotropía)')
print('- Manhattan: diamantes (movimiento por ejes)')
print('- Coseno: depende del ángulo, no de la distancia física')


---

## 4. Evaluación de Clustering

### <a id="coeficiente-de-silueta"></a>Coeficiente de Silueta (Silhouette)

**Definición:** Métrica que mide qué tan bien está asignado cada punto a su cluster. Combina **cohesión interna** (qué tan cerca está de su cluster) y **separación** (qué tan lejos está del cluster vecino más cercano).

**Fórmula para un punto $i$:**

$$
s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}
$$

donde:
- $a(i)$: distancia media de $i$ a los demás puntos de su **mismo** cluster (cohesión)
- $b(i)$: distancia media de $i$ al cluster **vecino más cercano** (separación)

**Rango:** $s(i) \in [-1, 1]$

| Valor | Interpretación |
|-------|----------------|
| **Cercano a 1** | Punto bien asignado — lejos del cluster vecino |
| **Cercano a 0** | Punto en la frontera — podría estar en cualquier cluster |
| **Negativo** | Punto mal asignado — más cerca del cluster vecino que del suyo |

**Silhouette promedio:**

$$
\bar{s} = \frac{1}{n} \sum_{i=1}^{n} s(i)
$$

Se usa para **elegir $K$**: el $K$ que maximiza $\bar{s}$ tiene clusters bien definidos.

**Ventajas:**
- ✅ Fácil de interpretar
- ✅ No requiere conocer las etiquetas verdaderas
- ✅ Funciona con cualquier métrica de distancia

**Limitaciones:**
- ❌ Costoso de calcular para datasets grandes (O(n²))
- ❌ Favorece clusters convexos y compactos

In [ ]:
# Ejemplo: Elegir K con Silhouette
from sklearn.metrics import silhouette_score

# Probamos diferentes valores de K
Ks = range(2, 8)
silhouettes = []

for k in Ks:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    score = silhouette_score(X, kmeans.labels_)
    silhouettes.append(score)

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(Ks, silhouettes, marker='o', linewidth=2, markersize=8, color='darkgreen')
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Silhouette promedio')
ax.set_title('Coeficiente de Silueta vs K')
ax.grid(True, alpha=0.3)
ax.axvline(Ks[np.argmax(silhouettes)], color='red', linestyle='--', 
           label=f'K óptimo = {Ks[np.argmax(silhouettes)]}')
ax.legend()
plt.show()

print(f'K con mayor silhouette: {Ks[np.argmax(silhouettes)]} (score = {max(silhouettes):.3f})')
print('Interpretación: Este K tiene los clusters mejor definidos (alta cohesión y separación)')


### <a id="metodo-del-codo"></a>Método del Codo (Elbow)

**Definición:** Heurística para elegir $K$ en K-Means graficando la **inercia** vs $K$ y buscando el "codo" — el punto donde la mejora marginal se aplana.

**Idea:**
- La inercia **siempre disminuye** al aumentar $K$
- Con $K=n$ (cada punto su propio cluster), inercia = 0
- Pero queremos el $K$ más pequeño que capture la estructura

**Cómo identificar el codo:**
1. Graficar inercia vs $K$
2. Buscar el punto donde la curva cambia de pendiente pronunciada a suave
3. Ese $K$ es un buen balance entre simplicidad y ajuste

**Ventajas:**
- ✅ Muy simple e intuitivo
- ✅ Rápido de calcular

**Limitaciones:**
- ❌ El "codo" no siempre es obvio (curva suave)
- ❌ Subjetivo — dos personas pueden elegir $K$ diferentes
- ❌ Solo aplica a K-Means (usa inercia)

**Recomendación:** Combinar con Silhouette para tener dos perspectivas independientes.

In [ ]:
# Ejemplo: Método del Codo
inercias = []

for k in Ks:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inercias.append(kmeans.inertia_)

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(Ks, inercias, marker='o', linewidth=2, markersize=8, color='blue')
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Inercia (suma de distancias al centroide²)')
ax.set_title('Método del Codo')
ax.grid(True, alpha=0.3)
plt.show()

print('Interpretación:')
print('- La inercia siempre disminuye al aumentar K')
print('- Busca el "codo" donde la curva se aplana')
print('- En este caso, el codo parece estar en K=3 o K=4')
print('- Después de ese punto, agregar más clusters no reduce mucho la inercia')


### <a id="davies-bouldin-index"></a>Davies-Bouldin Index

**Definición:** Ratio promedio de la dispersión intra-cluster vs la separación inter-cluster. **Menor es mejor**.

**Fórmula:**

$$
DB = \frac{1}{K} \sum_{i=1}^{K} \max_{j \neq i} \left( \frac{s_i + s_j}{d_{ij}} \right)
$$

donde $s_i$ es la dispersión promedio del cluster $i$ y $d_{ij}$ es la distancia entre centroides.

**Interpretación:** Mide qué tan "separados" están los clusters. Valores bajos indican clusters compactos y bien separados.

---

### <a id="calinski-harabasz-index"></a>Calinski-Harabasz Index

**Definición:** Ratio de la varianza entre-clusters vs la varianza intra-cluster. **Mayor es mejor**.

**Fórmula:**

$$
CH = \frac{\text{tr}(B_K)}{\text{tr}(W_K)} \cdot \frac{n - K}{K - 1}
$$

donde $B_K$ es la matriz de dispersión entre-clusters y $W_K$ es la intra-cluster.

**Interpretación:** Valores altos indican clusters bien definidos (mucha varianza entre clusters, poca dentro de cada uno).

**Uso:** Ambas métricas son alternativas a Silhouette, más rápidas de calcular pero menos intuitivas.

---

## 5. Reducción de Dimensionalidad

### <a id="pca"></a>PCA (Principal Component Analysis)

**Definición:** Técnica que encuentra un número pequeño de **nuevos ejes** (componentes principales) que capturan la mayor varianza posible de los datos. Es una transformación lineal que proyecta los datos a un espacio de menor dimensión.

**Objetivo:**

Dado $X \in \mathbb{R}^{n \times p}$, encontrar $k < p$ ejes ortogonales tales que la proyección sobre esos ejes **maximice la varianza** (o equivalentemente, **minimice el error de reconstrucción**).

**Formulación matemática:**

Los componentes principales son los **autovectores** de la matriz de covarianza $\Sigma = \frac{1}{n-1} X^\top X$, ordenados por sus **autovalores** $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_p$:

$$
\Sigma v_k = \lambda_k v_k
$$

- $v_k$: dirección del $k$-ésimo componente principal
- $\lambda_k$: varianza explicada por ese componente

**Proyección:**

$$
Z = X V_k \in \mathbb{R}^{n \times k}
$$

donde $V_k$ contiene los $k$ primeros autovectores.

**Casos de uso:**
- **Visualización:** Proyectar datos de alta dimensión a 2D o 3D
- **Preprocesamiento:** Reducir dimensión antes de clustering o clasificación
- **Compresión:** Almacenar datos con menos features
- **Eliminación de ruido:** Los últimos componentes suelen ser ruido
- **Detección de multicolinealidad:** Variables correlacionadas se "colapsan" en un PC

**Ventajas:**
- ✅ Reduce dimensión preservando máxima varianza
- ✅ Elimina correlaciones (componentes son ortogonales)
- ✅ Rápido y determinístico

**Limitaciones:**
- ❌ Solo captura relaciones **lineales**
- ❌ Los componentes son combinaciones de todas las variables (menos interpretables)
- ❌ Sensible a la escala → **siempre escalar antes**
- ❌ Sensible a outliers

In [ ]:
# Ejemplo visual: PCA en 2D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Generamos datos correlacionados (elipse)
np.random.seed(42)
mean = [0, 0]
cov = [[3, 1.8], [1.8, 1]]  # Covarianza alta → variables correlacionadas
X_pca = np.random.multivariate_normal(mean, cov, size=300)

# Aplicamos PCA
pca = PCA(n_components=2).fit(X_pca)
comps = pca.components_  # Direcciones de los componentes
expl_var = pca.explained_variance_  # Varianza de cada componente

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Izquierda: Datos originales con ejes de PCA
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=20)
for i, (vec, var) in enumerate(zip(comps, expl_var)):
    v = vec * np.sqrt(var) * 2.5  # Escala para visualizar
    axes[0].annotate('', xy=v, xytext=(0, 0),
                    arrowprops={'arrowstyle': '->', 'lw': 3, 'color': 'red'})
    axes[0].text(v[0]*1.15, v[1]*1.15, f'PC{i+1}\n({pca.explained_variance_ratio_[i]*100:.1f}%)',
                fontsize=12, color='red', fontweight='bold', ha='center')
axes[0].set_aspect('equal')
axes[0].set_title('Datos originales + Componentes Principales')
axes[0].set_xlabel('x₁')
axes[0].set_ylabel('x₂')
axes[0].grid(True, alpha=0.3)

# Derecha: Datos proyectados en espacio PCA
Z = pca.transform(X_pca)
axes[1].scatter(Z[:, 0], Z[:, 1], alpha=0.5, s=20, color='green')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[1].axvline(0, color='black', linestyle='--', alpha=0.3)
axes[1].set_aspect('equal')
axes[1].set_title('Datos en espacio PCA (decorrelacionados)')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Interpretación:')
print('- PC1 (flecha roja más larga): dirección de máxima varianza')
print('- PC2 (flecha roja más corta): dirección ortogonal con segunda mayor varianza')
print('- En el espacio PCA, los ejes están decorrelacionados (nube circular)')


### <a id="componentes-principales"></a>Componentes Principales

**Definición:** Nuevos ejes (direcciones) en el espacio de los datos que capturan la máxima varianza. Son combinaciones lineales de las variables originales.

**Propiedades:**
- **Ortogonales:** Los componentes son perpendiculares entre sí (no correlacionados)
- **Ordenados:** PC1 captura más varianza que PC2, que captura más que PC3, etc.
- **Combinaciones lineales:** $PC_k = w_{k1} x_1 + w_{k2} x_2 + \dots + w_{kp} x_p$

**Ejemplo:** Si tienes datos de clientes con `tenure`, `MonthlyCharges`, `TotalCharges`:
- PC1 podría ser "antigüedad + gasto" (clientes leales de alto valor)
- PC2 podría ser "gasto mensual alto pero tenure bajo" (clientes premium recientes)

---

### <a id="varianza-explicada"></a>Varianza Explicada

**Definición:** Proporción de la varianza total de los datos que captura cada componente principal.

**Fórmula:**

$$
\text{ratio}_k = \frac{\lambda_k}{\sum_{j=1}^{p} \lambda_j}
$$

**Varianza acumulada:**

$$
\text{acum}_k = \sum_{j=1}^{k} \text{ratio}_j
$$

**Cómo elegir $k$:**
1. **Umbral:** Quedarse con los componentes que acumulen 80-95% de la varianza
2. **Regla de Kaiser:** Solo componentes con $\lambda_k > 1$ (cuando los datos están estandarizados)
3. **Scree plot:** Buscar el "codo" donde la varianza se aplana

**Ejemplo:** Si PC1 explica 60%, PC2 explica 25%, PC3 explica 10%:
- Con 2 componentes capturas 85% de la varianza
- Pierdes solo 15% de información pero reduces de $p$ a 2 dimensiones

### <a id="loadings"></a>Loadings

**Definición:** Pesos (coeficientes) de cada variable original en cada componente principal. Indican **qué variables contribuyen más** a cada PC.

**Interpretación:**
- **Loading alto positivo:** La variable empuja la observación hacia el lado positivo del PC
- **Loading alto negativo:** Empuja hacia el lado negativo
- **Loading cercano a 0:** La variable no contribuye a ese PC

**Ejemplo:**

Si PC1 tiene loadings:
- `tenure`: 0.6
- `MonthlyCharges`: 0.5
- `TotalCharges`: 0.6

Entonces PC1 representa "antigüedad y gasto total" (todas positivas, similares).

**Uso práctico:** Los loadings permiten **interpretar** qué significa cada componente en términos de las variables originales.

---

### <a id="scree-plot"></a>Scree Plot

**Definición:** Gráfico de la varianza explicada por cada componente vs el número de componente. Ayuda a decidir cuántos componentes conservar.

**Cómo leerlo:**
- Eje X: Número de componente (1, 2, 3, ...)
- Eje Y: Varianza explicada (autovalor $\lambda_k$)
- Buscar el "codo" donde la curva se aplana

**Interpretación:** Después del codo, los componentes adicionales aportan poco (probablemente ruido).

---

### <a id="maldicion-de-la-dimension"></a>Maldición de la Dimensión

**Definición:** Fenómeno donde, al aumentar el número de dimensiones $p$, las distancias entre puntos se vuelven similares y los algoritmos basados en distancia pierden poder discriminativo.

**Problemas:**
- Las distancias eucl idianas se concentran (todos los puntos están "lejos")
- El volumen del espacio crece exponencialmente → datos muy esparsos
- Necesitas exponencialmente más datos para mantener la densidad

**Solución:** Reducción de dimensionalidad (PCA, t-SNE, UMAP) antes de clustering o clasificación.

**Ejemplo:** Con 10 features y 1000 observaciones, tienes buena densidad. Con 100 features, los mismos 1000 puntos están muy dispersos en el espacio.

---

## 6. Reglas de Asociación

### <a id="market-basket-analysis"></a>Market Basket Analysis

**Definición:** Técnica para descubrir patrones de co-ocurrencia en transacciones. Busca reglas del tipo "quien compra A también compra B".

**Vocabulario:**
- **Ítem:** Producto individual (pan, leche, café)
- **Itemset:** Conjunto de ítems ({pan, leche})
- **Transacción:** Conjunto de ítems comprados en un evento
- **Regla:** $A \Rightarrow B$ donde $A$ (antecedente) y $B$ (consecuente) son itemsets disjuntos

**Casos de uso:**
- **Retail:** Diseño de góndolas, promociones cruzadas
- **E-commerce:** Recomendaciones ("clientes que vieron X también vieron Y")
- **Salud:** Síntomas que aparecen juntos
- **Web analytics:** Secuencias de clicks

---

### <a id="soporte"></a>Soporte (Support)

**Definición:** Frecuencia con la que un itemset aparece en las transacciones.

**Fórmula:**

$$
\text{sop}(A) = \frac{|\{T \in D : A \subseteq T\}|}{|D|}
$$

**Interpretación:** Proporción de transacciones que contienen $A$.

**Ejemplo:** Si 150 de 1000 transacciones contienen {pan, leche}:
$$\text{sop}(\{\text{pan}, \text{leche}\}) = \frac{150}{1000} = 0.15 = 15\%$$

**Uso:** Filtrar itemsets raros con `min_support` (e.g., 5%). Itemsets muy raros no son útiles para recomendaciones masivas.

---

### <a id="confianza"></a>Confianza (Confidence)

**Definición:** Probabilidad condicional de comprar $B$ dado que se compró $A$.

**Fórmula:**

$$
\text{conf}(A \Rightarrow B) = \frac{\text{sop}(A \cup B)}{\text{sop}(A)} = P(B | A)
$$

**Interpretación:** De las transacciones que contienen $A$, ¿qué proporción también contiene $B$?

**Ejemplo:** Si 150 transacciones tienen {pan, leche} y 120 de esas también tienen {mantequilla}:
$$\text{conf}(\{\text{pan}, \text{leche}\} \Rightarrow \{\text{mantequilla}\}) = \frac{120}{150} = 0.8 = 80\%$$

**Limitación:** Alta confianza no garantiza una regla útil. Si $B$ es muy popular (e.g., bolsas plásticas), la confianza será alta aunque no haya asociación real. Por eso se usa **lift**.

### <a id="lift"></a>Lift

**Definición:** Cuánto más probable es comprar $B$ junto con $A$ comparado con comprar $B$ solo (independientemente).

**Fórmula:**

$$
\text{lift}(A \Rightarrow B) = \frac{\text{conf}(A \Rightarrow B)}{\text{sop}(B)} = \frac{\text{sop}(A \cup B)}{\text{sop}(A) \cdot \text{sop}(B)}
$$

**Interpretación:**

| Lift | Significado |
|------|-------------|
| **> 1** | $A$ y $B$ se compran juntos **más** que por azar → regla útil |
| **= 1** | Independientes → la regla no aporta información |
| **< 1** | Se evitan entre sí (productos sustitutos) |

**Ejemplo:** 
- $\text{sop}(\{\text{pan}, \text{leche}\}) = 0.15$
- $\text{sop}(\{\text{pan}\}) = 0.30$
- $\text{sop}(\{\text{leche}\}) = 0.40$

$$\text{lift} = \frac{0.15}{0.30 \times 0.40} = \frac{0.15}{0.12} = 1.25$$

Interpretación: Comprar pan y leche juntos es 1.25× más probable que si fueran independientes.

**Uso práctico:** Filtrar reglas con `lift >= 1.2` para quedarse solo con asociaciones fuertes.

> 💡 **Importante:** Una regla con alta confianza pero lift ≈ 1 es engañosa — ocurre solo porque $B$ es muy popular, no porque haya asociación real con $A$.

### <a id="algoritmo-apriori"></a>Algoritmo Apriori

**Definición:** Algoritmo clásico para encontrar itemsets frecuentes y generar reglas de asociación. Usa la **propiedad anti-monótona del soporte** para podar el espacio de búsqueda.

**Propiedad anti-monótona:**

> "Si un itemset $A$ es infrecuente (soporte < `min_support`), ningún superconjunto de $A$ puede ser frecuente."

**Algoritmo:**
1. **k=1:** Encontrar todos los ítems individuales con soporte ≥ `min_support`
2. **k=2:** Combinar pares de ítems frecuentes, contar soporte, filtrar
3. **k=3, k=4, ...:** Combinar itemsets de tamaño $k$ frecuentes para generar candidatos de tamaño $k+1$
4. **Detener:** Cuando no haya más candidatos frecuentes
5. **Generar reglas:** A partir de los itemsets frecuentes, crear reglas y filtrar por `min_confidence` y `min_lift`

**Ventajas:**
- ✅ Garantiza encontrar todos los itemsets frecuentes
- ✅ Poda eficiente del espacio de búsqueda

**Limitaciones:**
- ❌ Genera muchos candidatos (costoso en memoria)
- ❌ Múltiples pasadas sobre los datos
- ❌ No escala bien a millones de transacciones

**Parámetros:**
- `min_support`: Umbral de frecuencia mínima (e.g., 5%)
- `min_confidence`: Umbral de confianza mínima (e.g., 40%)
- `min_lift`: Umbral de lift mínimo (e.g., 1.2)

---

### <a id="fp-growth"></a>FP-Growth

**Definición:** Algoritmo más eficiente que Apriori. Comprime las transacciones en un **árbol FP (Frequent Pattern tree)** y extrae itemsets frecuentes sin generar candidatos explícitos.

**Ventajas sobre Apriori:**
- ✅ Más rápido (1-2 órdenes de magnitud)
- ✅ Solo 2 pasadas sobre los datos
- ✅ No genera candidatos explícitos
- ✅ Usa menos memoria

**Cuándo usar:**
- Datasets grandes (millones de transacciones)
- Catálogos grandes (miles de ítems)
- Cuando Apriori es muy lento

**API en Python:** Idéntica a Apriori (mlxtend):
```python
from mlxtend.frequent_patterns import fpgrowth
itemsets = fpgrowth(basket, min_support=0.05, use_colnames=True)
```

In [ ]:
# Ejemplo: Reglas de Asociación (sintético)
import pandas as pd

# Generamos transacciones sintéticas
np.random.seed(42)
productos = ['pan', 'leche', 'mantequilla', 'cafe', 'azucar', 'huevos', 'queso']
n_trans = 200

# Creamos asociaciones intencionales
transacciones = []
for _ in range(n_trans):
    trans = set()
    # 40% probabilidad: pan + mantequilla
    if np.random.rand() < 0.4:
        trans.update(['pan', 'mantequilla'])
    # 30% probabilidad: cafe + azucar
    if np.random.rand() < 0.3:
        trans.update(['cafe', 'azucar'])
    # Agregar 1-3 productos aleatorios
    trans.update(np.random.choice(productos, size=np.random.randint(1, 4), replace=False))
    transacciones.append(trans)

# Convertir a formato one-hot (basket)
basket = pd.DataFrame([{p: (p in t) for p in productos} for t in transacciones])

print('Primeras 5 transacciones:')
print(basket.head())
print(f'\nTotal de transacciones: {len(basket)}')


In [ ]:
# Aplicar Apriori (requiere mlxtend: uv add mlxtend)
try:
    from mlxtend.frequent_patterns import apriori, association_rules
    
    # Encontrar itemsets frecuentes
    itemsets = apriori(basket, min_support=0.1, use_colnames=True)
    
    # Generar reglas
    reglas = association_rules(itemsets, metric='confidence', min_threshold=0.3)
    reglas = reglas[reglas['lift'] >= 1.0]
    
    # Formatear para visualización
    reglas['antecedents'] = reglas['antecedents'].apply(lambda x: ', '.join(list(x)))
    reglas['consequents'] = reglas['consequents'].apply(lambda x: ', '.join(list(x)))
    
    print('Top 10 reglas por lift:')
    print(reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
          .sort_values('lift', ascending=False)
          .head(10)
          .round(3)
          .to_string(index=False))
    
    print('\nInterpretación:')
    print('- Reglas con lift > 1: asociaciones reales (no por azar)')
    print('- Alta confianza: si compras A, es muy probable que compres B')
    print('- Soporte: qué tan frecuente es la combinación')
    
except ImportError:
    print('mlxtend no está instalado. Instala con: uv add mlxtend')
    print('Las reglas esperadas serían:')
    print('- pan → mantequilla (lift alto)')
    print('- cafe → azucar (lift alto)')
    print('Estas asociaciones las "sembramos" en los datos sintéticos')


---

## Referencias y Recursos Adicionales

### Libros
- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*, Cap. 14
- James, G. et al. (2021). *An Introduction to Statistical Learning* (ISLR), Cap. 12
- Aggarwal, C. C. (2015). *Data Mining: The Textbook*

### Papers Clásicos
- Lloyd, S. (1982). *Least squares quantization in PCM* (K-Means)
- Pearson, K. (1901). *On lines and planes of closest fit* (PCA)
- Agrawal, R. & Srikant, R. (1994). *Fast Algorithms for Mining Association Rules* (Apriori)
- Rousseeuw, P. (1987). *Silhouettes: A graphical aid* (Silhouette)

### Documentación
- scikit-learn Clustering: https://scikit-learn.org/stable/modules/clustering.html
- scikit-learn Decomposition: https://scikit-learn.org/stable/modules/decomposition.html
- mlxtend Frequent Patterns: https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/

### Datasets
- Telco Customer Churn: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- Online Retail (UCI): https://archive.ics.uci.edu/dataset/352/online+retail

---

## Resumen de Cuándo Usar Cada Técnica

| Técnica | Cuándo usar | Ejemplo |
|---------|-------------|---------|
| **K-Means** | Segmentación rápida, clusters esféricos | Segmentar clientes por comportamiento |
| **Clustering Jerárquico** | Explorar estructura, no sabes K | Agrupar genes por expresión similar |
| **PCA** | Reducir dimensión, visualizar, eliminar correlaciones | Visualizar clusters en 2D, preprocesar antes de clustering |
| **Reglas de Asociación** | Descubrir co-ocurrencias, recomendaciones | "Quien compra X también compra Y" |

---

> 📚 **Nota final:** Este glosario cubre los conceptos fundamentales del módulo 2. Para profundizar, consulta los notebooks específicos:
> - `01_introduccion_aprendizaje_no_supervisado.ipynb`
> - `02_clustering_kmeans_jerarquico.ipynb`
> - `03_reduccion_dimensionalidad_pca.ipynb`
> - `04_reglas_asociacion.ipynb`